# Accepted Loan Preprocessing: Split, Impute, Encode

This notebook consumes the exported Cleaning datasets and prepares model-ready baseline and challenger matrices for binary `target_bad` prediction.

It intentionally fits all preprocessing decisions on the training split only to avoid validation/test leakage.

## 1. Configuration

Define project paths, output folders, split proportions, baseline/challenger feature policy, and reproducibility settings.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/lendingclub_mplconfig")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
TEST_SHARE = 0.15

DEFAULT_PROJECT_ROOT = Path(
    "/Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/"
    "Final_Project/Final/CreditRiskRAG"
)

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "Cleaning").exists() and (candidate / "README.md").exists():
            return candidate
    return DEFAULT_PROJECT_ROOT

PROJECT_ROOT = find_project_root()
CLEANING_DATASET_DIR = PROJECT_ROOT / "Cleaning" / "cleaning_outputs" / "datasets"
PREPROCESSING_OUTPUT_ROOT = PROJECT_ROOT / "Preprocessing" / "preprocessing_outputs"
TABLE_DIR = PREPROCESSING_OUTPUT_ROOT / "tables"
DATASET_DIR = PREPROCESSING_OUTPUT_ROOT / "datasets"
PLOT_DIR = PREPROCESSING_OUTPUT_ROOT / "plots"
for path in [TABLE_DIR, DATASET_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

X_PATH = CLEANING_DATASET_DIR / "accepted_X_starter.parquet"
Y_PATH = CLEANING_DATASET_DIR / "accepted_y_target_bad.parquet"
TRACE_PATH = CLEANING_DATASET_DIR / "accepted_traceability.parquet"

BASELINE_EXCLUDED_FEATURES = ["mths_since_last_record"]
BASELINE_MISSING_INDICATOR_FEATURES = ["mths_since_last_delinq", "emp_length_years"]
CHALLENGER_EXTRA_MISSING_INDICATOR_FEATURES = ["mths_since_last_record"]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Cleaning datasets:", CLEANING_DATASET_DIR)
print("Preprocessing outputs:", PREPROCESSING_OUTPUT_ROOT)

PROJECT_ROOT: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG
Cleaning datasets: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Cleaning/cleaning_outputs/datasets
Preprocessing outputs: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs


## 2. Load Cleaned Inputs

Load the starter feature matrix, target vector, and traceability frame exported by Cleaning.

In [2]:
def save_table(df: pd.DataFrame, name: str, index: bool = False) -> Path:
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=index)
    print("Saved:", path)
    return path

for path in [X_PATH, Y_PATH, TRACE_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required cleaning output: {path}")

X = pd.read_parquet(X_PATH)
y = pd.read_parquet(Y_PATH)
traceability = pd.read_parquet(TRACE_PATH)

if len(X) != len(y) or len(X) != len(traceability):
    raise ValueError(f"Input row mismatch: X={len(X)}, y={len(y)}, traceability={len(traceability)}")
if "target_bad" not in y.columns:
    raise ValueError("Expected target column target_bad in y dataset")
if "issue_d_dt" not in traceability.columns:
    raise ValueError("Expected issue_d_dt in traceability dataset for chronological split")

y_series = y["target_bad"].astype(int)
traceability = traceability.copy()
traceability["issue_d_dt"] = pd.to_datetime(traceability["issue_d_dt"], errors="coerce")
if traceability["issue_d_dt"].isna().any():
    raise ValueError("issue_d_dt contains missing or invalid dates")

input_summary = pd.DataFrame([
    {"item": "rows", "value": len(X)},
    {"item": "starter_features", "value": X.shape[1]},
    {"item": "target_bad_rate", "value": round(float(y_series.mean()), 6)},
    {"item": "issue_date_min", "value": traceability["issue_d_dt"].min().date().isoformat()},
    {"item": "issue_date_max", "value": traceability["issue_d_dt"].max().date().isoformat()},
])
save_table(input_summary, "preprocessing_input_summary")
display(input_summary)
display(X.head())

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/tables/preprocessing_input_summary.csv


,item,value
0,rows,1345310
1,starter_features,38
2,target_bad_rate,0.199626
3,issue_date_min,2007-06-01
4,issue_date_max,2018-12-01


,loan_amnt,term_months,int_rate_clean,installment,grade,sub_grade,emp_length_years,home_ownership,annual_inc,verification_status,purpose,dti,delinq_2yrs,fico_mean,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util_clean,total_acc,credit_history_years,collections_12_mths_ex_med,acc_now_delinq,tot_coll_amt,tot_cur_bal,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,mort_acc,pub_rec_bankruptcies,tax_liens,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,application_type
0,3600.0,36,13.99,123.03,C,C4,10.0,MORTGAGE,55000.0,Not Verified,debt_consolidation,5.91,0.0,677.0,1.0,30.0,NaN,7.0,0.0,2765.0,29.7,13.0,12.334018,0.0,0.0,722.0,144904.0,4.0,20701.0,1506.0,37.2,1.0,0.0,0.0,7746.0,2400.0,13734.0,Individual
1,24700.0,36,11.99,820.28,C,C1,10.0,MORTGAGE,65000.0,Not Verified,small_business,16.06,1.0,717.0,4.0,6.0,NaN,22.0,0.0,21470.0,19.2,38.0,16.000000,0.0,0.0,0.0,204396.0,4.0,9733.0,57830.0,27.1,4.0,0.0,0.0,39475.0,79300.0,24667.0,Individual
2,20000.0,60,10.78,432.66,B,B4,10.0,MORTGAGE,63000.0,Not Verified,home_improvement,10.78,0.0,697.0,0.0,NaN,NaN,6.0,0.0,7869.0,56.2,18.0,15.331964,0.0,0.0,0.0,189699.0,6.0,31617.0,2737.0,55.9,5.0,0.0,0.0,18696.0,6200.0,14877.0,Joint App
3,10400.0,60,22.45,289.91,F,F1,3.0,MORTGAGE,104433.0,Source Verified,major_purchase,25.37,1.0,697.0,3.0,12.0,NaN,12.0,0.0,21929.0,64.5,35.0,17.500342,0.0,0.0,0.0,331730.0,10.0,27644.0,4567.0,77.5,6.0,0.0,0.0,95768.0,20300.0,88097.0,Individual
4,11950.0,36,13.44,405.18,C,C3,4.0,RENT,34000.0,Source Verified,debt_consolidation,10.20,0.0,692.0,0.0,NaN,NaN,5.0,0.0,8822.0,68.4,6.0,28.167009,0.0,0.0,0.0,12798.0,0.0,2560.0,844.0,91.0,0.0,0.0,0.0,12798.0,9400.0,4000.0,Individual


## 3. Chronological Split

Create train, validation, and test splits by `issue_d_dt`. No imputation, encoding, category discovery, or scaling is fit before this split.

In [3]:
def chronological_split_indices(issue_dates: pd.Series, train_share: float, validation_share: float) -> tuple[pd.Index, pd.Index, pd.Index, pd.Timestamp, pd.Timestamp]:
    ordered = issue_dates.sort_values(kind="mergesort")
    n = len(ordered)
    train_end_pos = max(1, min(n - 2, int(np.floor(n * train_share))))
    valid_end_pos = max(train_end_pos + 1, min(n - 1, int(np.floor(n * (train_share + validation_share)))))

    train_cutoff = ordered.iloc[train_end_pos - 1]
    validation_cutoff = ordered.iloc[valid_end_pos - 1]

    train_idx = issue_dates[issue_dates <= train_cutoff].index
    validation_idx = issue_dates[(issue_dates > train_cutoff) & (issue_dates <= validation_cutoff)].index
    test_idx = issue_dates[issue_dates > validation_cutoff].index
    return train_idx, validation_idx, test_idx, train_cutoff, validation_cutoff

train_idx, validation_idx, test_idx, train_cutoff, validation_cutoff = chronological_split_indices(
    traceability["issue_d_dt"], TRAIN_SHARE, VALIDATION_SHARE
)

split_labels = pd.Series(index=X.index, dtype="string")
split_labels.loc[train_idx] = "train"
split_labels.loc[validation_idx] = "validation"
split_labels.loc[test_idx] = "test"

split_summary = (
    pd.DataFrame({"split": split_labels, "target_bad": y_series, "issue_d_dt": traceability["issue_d_dt"]})
    .groupby("split", observed=True)
    .agg(
        rows=("target_bad", "size"),
        bad_rate=("target_bad", "mean"),
        issue_min=("issue_d_dt", "min"),
        issue_max=("issue_d_dt", "max"),
    )
    .reset_index()
)
split_summary["row_pct"] = (split_summary["rows"] / len(X) * 100).round(4)
split_summary["bad_rate"] = split_summary["bad_rate"].round(6)
split_summary["issue_min"] = split_summary["issue_min"].dt.date.astype(str)
split_summary["issue_max"] = split_summary["issue_max"].dt.date.astype(str)

split_rule = pd.DataFrame([
    {"split": "train", "date_rule": f"issue_d_dt <= {train_cutoff.date()}"},
    {"split": "validation", "date_rule": f"{train_cutoff.date()} < issue_d_dt <= {validation_cutoff.date()}"},
    {"split": "test", "date_rule": f"issue_d_dt > {validation_cutoff.date()}"},
])

save_table(split_rule, "preprocessing_chronological_split_rule")
save_table(split_summary, "preprocessing_split_summary")
display(split_rule)
display(split_summary)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/tables/preprocessing_chronological_split_rule.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/tables/preprocessing_split_summary.csv


,split,date_rule
0,train,issue_d_dt <= 2016-04-01
1,validation,2016-04-01 < issue_d_dt <= 2017-02-01
2,test,issue_d_dt > 2017-02-01


,split,rows,bad_rate,issue_min,issue_max,row_pct
0,test,195749,0.210315,2017-03-01,2018-12-01,14.5505
1,train,962641,0.188300,2007-06-01,2016-04-01,71.5553
2,validation,186920,0.246763,2016-05-01,2017-02-01,13.8942


## 4. Feature Set Definitions

Build two feature sets:

- Baseline: excludes `mths_since_last_record` because it is about 83% missing.
- Challenger: includes `mths_since_last_record` with a missingness indicator.

In [4]:
baseline_features = [c for c in X.columns if c not in BASELINE_EXCLUDED_FEATURES]
challenger_features = list(X.columns)

feature_set_summary = pd.DataFrame([
    {"feature_set": "baseline", "feature_count": len(baseline_features), "excluded_features": "; ".join(BASELINE_EXCLUDED_FEATURES)},
    {"feature_set": "missingness_challenger", "feature_count": len(challenger_features), "excluded_features": ""},
])
save_table(feature_set_summary, "preprocessing_feature_set_summary")
display(feature_set_summary)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/tables/preprocessing_feature_set_summary.csv


,feature_set,feature_count,excluded_features
0,baseline,37,mths_since_last_record
1,missingness_challenger,38,


## 5. Train-Fit Preprocessing Functions

Fit medians, category levels, and scaling statistics on train only. Apply the learned preprocessing to validation and test without refitting.

In [5]:
def infer_feature_types(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    categorical_cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) or pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c])]
    numeric_cols = [c for c in df.columns if c not in categorical_cols]
    return numeric_cols, categorical_cols


def fit_preprocessor(
    X_train: pd.DataFrame,
    missing_indicator_features: list[str],
    scale_numeric: bool = True,
) -> dict:
    numeric_cols, categorical_cols = infer_feature_types(X_train)
    train_numeric = X_train[numeric_cols].apply(pd.to_numeric, errors="coerce")
    numeric_medians = train_numeric.median().fillna(0.0)
    numeric_means = train_numeric.fillna(numeric_medians).mean()
    numeric_stds = train_numeric.fillna(numeric_medians).std(ddof=0).replace(0, 1.0).fillna(1.0)
    categories = {
        col: sorted(X_train[col].astype("string").fillna("Missing").unique().tolist())
        for col in categorical_cols
    }
    indicators = [c for c in missing_indicator_features if c in X_train.columns]
    return {
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "numeric_medians": numeric_medians,
        "numeric_means": numeric_means,
        "numeric_stds": numeric_stds,
        "categories": categories,
        "missing_indicator_features": indicators,
        "scale_numeric": scale_numeric,
    }


def transform_with_preprocessor(X_part: pd.DataFrame, preprocessor: dict) -> pd.DataFrame:
    pieces = []

    indicator_cols = {}
    for col in preprocessor["missing_indicator_features"]:
        if col in X_part.columns:
            indicator_cols[f"{col}_is_missing"] = X_part[col].isna().astype("int8")
    if indicator_cols:
        pieces.append(pd.DataFrame(indicator_cols, index=X_part.index))

    numeric_cols = preprocessor["numeric_cols"]
    if numeric_cols:
        numeric = X_part[numeric_cols].apply(pd.to_numeric, errors="coerce")
        numeric = numeric.fillna(preprocessor["numeric_medians"])
        if preprocessor["scale_numeric"]:
            numeric = (numeric - preprocessor["numeric_means"]) / preprocessor["numeric_stds"]
        numeric = numeric.astype("float32")
        pieces.append(numeric)

    for col in preprocessor["categorical_cols"]:
        values = X_part[col].astype("string").fillna("Missing")
        dtype = pd.CategoricalDtype(categories=preprocessor["categories"][col])
        values = values.astype(dtype)
        dummies = pd.get_dummies(values, prefix=col, dummy_na=False, dtype="int8")
        pieces.append(dummies)

    if not pieces:
        return pd.DataFrame(index=X_part.index)
    return pd.concat(pieces, axis=1)


def fit_transform_feature_set(feature_set_name: str, features: list[str], missing_indicator_features: list[str]) -> tuple[dict, dict[str, pd.DataFrame], pd.DataFrame]:
    X_subset = X[features].copy()
    X_train = X_subset.loc[train_idx]
    preprocessor = fit_preprocessor(X_train, missing_indicator_features=missing_indicator_features, scale_numeric=True)

    transformed = {
        "train": transform_with_preprocessor(X_subset.loc[train_idx], preprocessor),
        "validation": transform_with_preprocessor(X_subset.loc[validation_idx], preprocessor),
        "test": transform_with_preprocessor(X_subset.loc[test_idx], preprocessor),
    }

    feature_rows = []
    for col in preprocessor["numeric_cols"]:
        feature_rows.append({"feature_set": feature_set_name, "feature": col, "role": "numeric_scaled", "source": col})
    for col in preprocessor["categorical_cols"]:
        feature_rows.append({"feature_set": feature_set_name, "feature": col, "role": "categorical_one_hot_source", "source": col})
    for col in preprocessor["missing_indicator_features"]:
        feature_rows.append({"feature_set": feature_set_name, "feature": f"{col}_is_missing", "role": "missing_indicator", "source": col})

    feature_manifest = pd.DataFrame(feature_rows)
    return preprocessor, transformed, feature_manifest

## 6. Baseline Preprocessing

Fit preprocessing on the train split for the baseline feature set. This excludes `mths_since_last_record`.

In [6]:
baseline_preprocessor, baseline_transformed, baseline_manifest = fit_transform_feature_set(
    feature_set_name="baseline",
    features=baseline_features,
    missing_indicator_features=BASELINE_MISSING_INDICATOR_FEATURES,
)

baseline_shapes = pd.DataFrame([
    {"feature_set": "baseline", "split": split, "rows": frame.shape[0], "columns": frame.shape[1], "missing_values": int(frame.isna().sum().sum())}
    for split, frame in baseline_transformed.items()
])
save_table(baseline_shapes, "preprocessing_baseline_shapes")
save_table(baseline_manifest, "preprocessing_baseline_feature_manifest")
display(baseline_shapes)
display(baseline_manifest.head(30))

/var/folders/7q/522tn7j14w119bz944bz8qy80000gn/T/ipykernel_29584/1886451521.py:2: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  categorical_cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) or pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c])]


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/tables/preprocessing_baseline_shapes.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/tables/preprocessing_baseline_feature_manifest.csv


,feature_set,split,rows,columns,missing_values
0,baseline,train,962641,100,0
1,baseline,validation,186920,100,0
2,baseline,test,195749,100,0


,feature_set,feature,role,source
0,baseline,loan_amnt,numeric_scaled,loan_amnt
1,baseline,term_months,numeric_scaled,term_months
2,baseline,int_rate_clean,numeric_scaled,int_rate_clean
3,baseline,installment,numeric_scaled,installment
4,baseline,emp_length_years,numeric_scaled,emp_length_years
5,baseline,annual_inc,numeric_scaled,annual_inc
6,baseline,dti,numeric_scaled,dti
7,baseline,delinq_2yrs,numeric_scaled,delinq_2yrs
8,baseline,fico_mean,numeric_scaled,fico_mean
9,baseline,inq_last_6mths,numeric_scaled,inq_last_6mths


## 7. Missingness Challenger Preprocessing

Fit preprocessing on the train split for the challenger feature set. This includes `mths_since_last_record` and adds its missingness indicator.

In [7]:
challenger_preprocessor, challenger_transformed, challenger_manifest = fit_transform_feature_set(
    feature_set_name="missingness_challenger",
    features=challenger_features,
    missing_indicator_features=BASELINE_MISSING_INDICATOR_FEATURES + CHALLENGER_EXTRA_MISSING_INDICATOR_FEATURES,
)

challenger_shapes = pd.DataFrame([
    {"feature_set": "missingness_challenger", "split": split, "rows": frame.shape[0], "columns": frame.shape[1], "missing_values": int(frame.isna().sum().sum())}
    for split, frame in challenger_transformed.items()
])
save_table(challenger_shapes, "preprocessing_challenger_shapes")
save_table(challenger_manifest, "preprocessing_challenger_feature_manifest")
display(challenger_shapes)
display(challenger_manifest.head(30))

/var/folders/7q/522tn7j14w119bz944bz8qy80000gn/T/ipykernel_29584/1886451521.py:2: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  categorical_cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) or pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c])]


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/tables/preprocessing_challenger_shapes.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/tables/preprocessing_challenger_feature_manifest.csv


,feature_set,split,rows,columns,missing_values
0,missingness_challenger,train,962641,102,0
1,missingness_challenger,validation,186920,102,0
2,missingness_challenger,test,195749,102,0


,feature_set,feature,role,source
0,missingness_challenger,loan_amnt,numeric_scaled,loan_amnt
1,missingness_challenger,term_months,numeric_scaled,term_months
2,missingness_challenger,int_rate_clean,numeric_scaled,int_rate_clean
3,missingness_challenger,installment,numeric_scaled,installment
4,missingness_challenger,emp_length_years,numeric_scaled,emp_length_years
5,missingness_challenger,annual_inc,numeric_scaled,annual_inc
6,missingness_challenger,dti,numeric_scaled,dti
7,missingness_challenger,delinq_2yrs,numeric_scaled,delinq_2yrs
8,missingness_challenger,fico_mean,numeric_scaled,fico_mean
9,missingness_challenger,inq_last_6mths,numeric_scaled,inq_last_6mths


## 8. Export Preprocessed Outputs

Save model-ready matrices, split targets, split traceability, and reusable preprocessing metadata. Generated matrices are large and should not be committed to Git.

In [8]:
def save_dataset(df: pd.DataFrame, stem: str) -> Path:
    path = DATASET_DIR / f"{stem}.parquet"
    df.to_parquet(path, index=True)
    print("Saved:", path)
    return path

export_rows = []

for feature_set_name, transformed in [
    ("baseline", baseline_transformed),
    ("missingness_challenger", challenger_transformed),
]:
    for split, frame in transformed.items():
        export_rows.append({
            "artifact": f"{feature_set_name}_{split}_X",
            "path": str(save_dataset(frame, f"{feature_set_name}_{split}_X")),
        })

for split, idx in [("train", train_idx), ("validation", validation_idx), ("test", test_idx)]:
    y_split = y_series.loc[idx].to_frame("target_bad")
    trace_split = traceability.loc[idx].copy()
    split_label_df = pd.DataFrame({"split": split_labels.loc[idx]}, index=idx)
    export_rows.append({"artifact": f"{split}_y", "path": str(save_dataset(y_split, f"{split}_y"))})
    export_rows.append({"artifact": f"{split}_traceability", "path": str(save_dataset(trace_split, f"{split}_traceability"))})
    export_rows.append({"artifact": f"{split}_labels", "path": str(save_dataset(split_label_df, f"{split}_labels"))})

preprocessing_metadata = {
    "train_share": TRAIN_SHARE,
    "validation_share": VALIDATION_SHARE,
    "test_share": TEST_SHARE,
    "train_cutoff": train_cutoff.date().isoformat(),
    "validation_cutoff": validation_cutoff.date().isoformat(),
    "baseline_excluded_features": BASELINE_EXCLUDED_FEATURES,
    "baseline_missing_indicator_features": BASELINE_MISSING_INDICATOR_FEATURES,
    "challenger_extra_missing_indicator_features": CHALLENGER_EXTRA_MISSING_INDICATOR_FEATURES,
    "baseline_numeric_columns": baseline_preprocessor["numeric_cols"],
    "baseline_categorical_columns": baseline_preprocessor["categorical_cols"],
    "challenger_numeric_columns": challenger_preprocessor["numeric_cols"],
    "challenger_categorical_columns": challenger_preprocessor["categorical_cols"],
}
metadata_path = TABLE_DIR / "preprocessing_metadata.json"
metadata_path.write_text(json.dumps(preprocessing_metadata, indent=2))
export_rows.append({"artifact": "preprocessing_metadata", "path": str(metadata_path)})

export_paths = pd.DataFrame(export_rows)
save_table(export_paths, "preprocessing_export_paths")
display(export_paths)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/baseline_train_X.parquet
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/baseline_validation_X.parquet


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/baseline_test_X.parquet


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/missingness_challenger_train_X.parquet
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/missingness_challenger_validation_X.parquet


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/missingness_challenger_test_X.parquet
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/train_y.parquet


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/train_traceability.parquet
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/train_labels.parquet
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/validation_y.parquet
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/validation_traceability.parquet
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/validation_labels.parquet
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Prepro

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/test_traceability.parquet
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/datasets/test_labels.parquet
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/tables/preprocessing_export_paths.csv


,artifact,path
0,baseline_train_X,/Users/lindaperez/Documents/NEU/2026SummerML/M...
1,baseline_validation_X,/Users/lindaperez/Documents/NEU/2026SummerML/M...
2,baseline_test_X,/Users/lindaperez/Documents/NEU/2026SummerML/M...
3,missingness_challenger_train_X,/Users/lindaperez/Documents/NEU/2026SummerML/M...
4,missingness_challenger_validation_X,/Users/lindaperez/Documents/NEU/2026SummerML/M...
5,missingness_challenger_test_X,/Users/lindaperez/Documents/NEU/2026SummerML/M...
6,train_y,/Users/lindaperez/Documents/NEU/2026SummerML/M...
7,train_traceability,/Users/lindaperez/Documents/NEU/2026SummerML/M...
8,train_labels,/Users/lindaperez/Documents/NEU/2026SummerML/M...
9,validation_y,/Users/lindaperez/Documents/NEU/2026SummerML/M...


## 9. Final QA

Confirm that preprocessed matrices have no missing values and that preprocessing was fit only from the train split.

In [9]:
qa_rows = []
for feature_set_name, transformed in [("baseline", baseline_transformed), ("missingness_challenger", challenger_transformed)]:
    train_columns = list(transformed["train"].columns)
    for split, frame in transformed.items():
        qa_rows.append({
            "feature_set": feature_set_name,
            "split": split,
            "rows": frame.shape[0],
            "columns": frame.shape[1],
            "missing_values": int(frame.isna().sum().sum()),
            "same_columns_as_train": list(frame.columns) == train_columns,
        })

preprocessing_qa = pd.DataFrame(qa_rows)
if (preprocessing_qa["missing_values"] != 0).any():
    raise AssertionError("Preprocessed matrices still contain missing values")
if not preprocessing_qa["same_columns_as_train"].all():
    raise AssertionError("Validation/test columns do not match train columns")

save_table(preprocessing_qa, "preprocessing_qa_summary")
display(preprocessing_qa)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Preprocessing/preprocessing_outputs/tables/preprocessing_qa_summary.csv


,feature_set,split,rows,columns,missing_values,same_columns_as_train
0,baseline,train,962641,100,0,True
1,baseline,validation,186920,100,0,True
2,baseline,test,195749,100,0,True
3,missingness_challenger,train,962641,102,0,True
4,missingness_challenger,validation,186920,102,0,True
5,missingness_challenger,test,195749,102,0,True
